# Laboratório 10: O Pipeline Definitivo
## RAG + QLoRA + FlashAttention + KV Cache

> **Partes deste laboratório foram geradas/complementadas com IA, revisadas e validadas por [Seu Nome]**

---
**Objetivo:** Orquestrar um pipeline de IA ponta a ponta para a HealthTech, resolvendo o problema de Out-Of-Memory (OOM) na GPU combinando quantização 4-bits (QLoRA), KV Cache e FlashAttention-2.

## 0. Instalação das Dependências

In [ ]:
# Instalar dependências necessárias
!pip install -q transformers>=4.40.0 bitsandbytes>=0.43.0 accelerate>=0.29.0
!pip install -q flash-attn --no-build-isolation  # Requer GPU com Ampere+ (ex: A100, RTX 3090+)

# Verificar ambiente
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA disponível: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1024**2
    print(f'VRAM total: {total_vram:.0f} MB')

## Passo 1: Ingestão Eficiente — Carregamento com QLoRA 4-bits

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import time

# --- Configuração QLoRA em 4-bits ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                          # Quantização 4-bits (QLoRA)
    bnb_4bit_compute_dtype=torch.float16,       # Computação em float16
    bnb_4bit_use_double_quant=True,             # Double quantization para mais compressão
    bnb_4bit_quant_type="nf4"                   # NormalFloat4: tipo de quantização
)

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("[1/2] Carregando tokenizador...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

# Zerar contadores de memória antes do carregamento
torch.cuda.reset_peak_memory_stats()
torch.cuda.empty_cache()
mem_antes = torch.cuda.memory_allocated() / 1024**2

print("[2/2] Carregando modelo com QLoRA 4-bits (sem FlashAttention ainda)...")
model_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model_base.eval()

# --- MÉTRICA 1: VRAM usada pelo modelo quantizado ---
mem_modelo_mb = (torch.cuda.memory_allocated() / 1024**2) - mem_antes
print(f"\n{'='*50}")
print(f"MÉTRICA 1 — VRAM ocupada pelo modelo (4-bits):")
print(f"  → {mem_modelo_mb:.2f} MB")
print(f"{'='*50}")

## Passo 2: Simulando o RAG Massivo (~10.000–15.000 tokens)

In [ ]:
import random

# --- Geração do contexto médico fictício ---
# Simulando 5 capítulos de manuais médicos recuperados pelo RAG

CAPITULOS_MEDICOS = [
    """CAPÍTULO 1 — FARMACOLOGIA CLÍNICA E MECANISMOS DE AÇÃO
A farmacologia clínica estuda os efeitos dos medicamentos no organismo humano e as interações
entre fármacos e receptores biológicos. Os mecanismos de ação envolvem a ligação a receptores
específicos, canais iônicos, enzimas e transportadores. A farmacocinética descreve a absorção,
distribuição, metabolismo e excreção (ADME). A absorção gastrointestinal é influenciada pelo
pH, lipofilia e transportadores de membrana. O metabolismo hepático de primeira passagem
reduz significativamente a biodisponibilidade oral de muitos fármacos. A excreção renal depende
da filtração glomerular, secreção tubular e reabsorção. Interações medicamentosas podem ocorrer
em qualquer etapa da farmacocinética ou farmacodinâmica. O índice terapêutico relaciona a dose
eficaz com a dose tóxica. Fármacos com índice terapêutico estreito, como a digoxina e a varfarina,
requerem monitorização plasmática rigorosa. Os receptores adrenérgicos alfa e beta modulam
respostas cardiovasculares e respiratórias. Agonistas beta-2 são fundamentais no tratamento da
asma brônquica, promovendo broncodilatação. Antagonistas beta-1 reduzem a frequência cardíaca
e a pressão arterial na insuficiência cardíaca e hipertensão. Os inibidores da ECA bloqueiam a
conversão da angiotensina I em angiotensina II, reduzindo a pós-carga cardíaca.""",

    """CAPÍTULO 2 — SEMIOLOGIA MÉDICA E EXAME CLÍNICO
A semiologia médica é a ciência dos sinais e sintomas das doenças. O exame clínico sistemático
inclui anamnese, inspeção, palpação, percussão e ausculta. A anamnese deve abordar queixa
principal, história da doença atual, antecedentes pessoais e familiares, medicamentos em uso,
alergias e hábitos de vida. Os sinais vitais incluem temperatura, pressão arterial, frequência
cardíaca, frequência respiratória e saturação de oxigênio. A febre é definida como temperatura
axilar acima de 37,8°C. A hipotensão ortostática é caracterizada por queda de 20 mmHg na
pressão sistólica ao levantar. A ausculta pulmonar revela crepitações, sibilos, roncos e atrito
pleural. Crepitações finas nas bases pulmonares sugerem edema pulmonar ou fibrose. Sibilos
difusos indicam broncoespasmo na asma ou DPOC. A ausculta cardíaca identifica bulhas, sopros
e atritos. Sopros sistólicos em foco aórtico podem indicar estenose aórtica. O abdômen deve ser
examinado em quatro quadrantes, avaliando dor, defesa e organomegalias.""",

    """CAPÍTULO 3 — URGÊNCIAS E EMERGÊNCIAS MÉDICAS
O atendimento de urgências e emergências exige rápida avaliação e intervenção. O protocolo
ABCDE (Airway, Breathing, Circulation, Disability, Exposure) é fundamental no trauma e na
instabilidade hemodinâmica. O choque é definido como hipoperfusão tecidual com oferta
inadequada de oxigênio. O choque hipovolêmico resulta de perda de volume intravascular;
o choque séptico decorre de infecção grave com resposta inflamatória sistêmica. O protocolo
Surviving Sepsis Campaign preconiza antibioticoterapia precoce em até 1 hora do diagnóstico.
A ressuscitação volêmica com cristaloides (soro fisiológico 0,9% ou Ringer Lactato) é indicada
no choque hemorrágico e séptico. O acidente vascular cerebral isquêmico exige tomografia
computadorizada imediata para exclusão de hemorragia antes da trombólise com rt-PA.
A janela terapêutica para trombólise é de 4,5 horas do início dos sintomas. O infarto agudo do
miocárdio com supradesnivelamento de ST (IAMCSST) requer angioplastia primária em até 90
minutos do primeiro contato médico. A hipoglicemia grave com Glasgow reduzido é tratada com
glicose hipertônica 50% intravenosa.""",

    """CAPÍTULO 4 — INTERPRETAÇÃO DE EXAMES LABORATORIAIS
Os exames laboratoriais fornecem informações objetivas para diagnóstico e acompanhamento.
O hemograma completo avalia série vermelha (eritrócitos, hemoglobina, hematócrito, VCM,
HCM, CHCM, RDW), série branca (leucócitos e diferencial) e plaquetas. A anemia é definida
como hemoglobina menor que 12 g/dL em mulheres e 13 g/dL em homens. Anemia microcítica
hipocrômica (VCM < 80 fL) sugere deficiência de ferro ou talassemia. Anemia macrocítica
(VCM > 100 fL) indica deficiência de B12 ou folato. A leucocitose com neutrofilia indica
processo infeccioso bacteriano. A neutropenia severa (< 500/mm³) aumenta drasticamente o
risco de infecções oportunistas. A troponina T de alta sensibilidade é o marcador padrão-ouro
para infarto do miocárdio. A creatinina sérica reflete a função renal; sua elevação indica
redução da taxa de filtração glomerular. A ureia sérica aumenta na insuficiência renal e em
estados catabólicos. A bilirrubina total fracionada distingue icterícia pré-hepática, hepática e
pós-hepática. O INR monitora a anticoagulação com varfarina; valores entre 2-3 são terapêuticos
para a maioria das indicações. A gasometria arterial avalia pH, pCO2, pO2, bicarbonato e
saturação de oxigênio, sendo essencial no diagnóstico de distúrbios ácido-base.""",

    """CAPÍTULO 5 — PROTOCOLOS TERAPÊUTICOS E CONDUTAS CLÍNICAS
Os protocolos terapêuticos padronizam condutas baseadas em evidências científicas de alto nível.
A hipertensão arterial sistêmica é tratada inicialmente com modificações no estilo de vida;
se insuficientes, iniciam-se anti-hipertensivos conforme perfil do paciente. Inibidores da ECA
ou BRAs são preferidos em diabéticos com proteinúria. Diuréticos tiazídicos são eficazes em
idosos. O diabetes mellitus tipo 2 tem como primeira linha a metformina, exceto em
contraindicações (DRC grave, insuficiência hepática). Inibidores de SGLT2 e agonistas de GLP-1
têm benefício cardiovascular e renal comprovados em estudos de desfecho. A dislipidemia
é tratada com estatinas de alta intensidade em pacientes de alto risco cardiovascular, visando
LDL < 70 mg/dL. A insuficiência cardíaca com fração de ejeção reduzida (ICFEr) é tratada com
a quádrupla terapia: IECA/BRA, betabloqueador, antagonista mineralocorticoide e inibidor de
SGLT2. A anticoagulação na fibrilação atrial não valvar visa CHA2DS2-VASc ≥ 2 em homens
e ≥ 3 em mulheres, preferencialmente com anticoagulantes orais diretos (DOACs)."""
]

# Expandir cada capítulo para aumentar o volume de tokens (~3x repetição + variações)
contexto_rag = ""
for i, cap in enumerate(CAPITULOS_MEDICOS):
    contexto_rag += cap + "\n\n"
    # Adicionar seções extras para atingir ~12.000 tokens
    contexto_rag += f"""Adicionalmente, o Capítulo {i+1} detalha aspectos complementares da prática clínica.
Estudos multicêntricos randomizados e controlados demonstram que a abordagem sistemática
e protocolizada melhora desfechos clínicos em até 35% comparada à conduta empírica isolada.
A educação continuada e a atualização constante dos profissionais de saúde são pilares
indispensáveis para a segurança do paciente e qualidade do atendimento hospitalar e ambulatorial.
Revisões sistemáticas e metanálises publicadas no NEJM, JAMA e Lancet consolidam as
recomendações baseadas em evidências que norteiam as diretrizes das sociedades médicas.
O uso racional de medicamentos, a farmacovigilância e a adesão ao tratamento constituem
desafios permanentes na gestão clínica moderna. Sistemas de apoio à decisão clínica integrados
aos prontuários eletrônicos auxiliam na prescrição segura e redução de erros médicos.\n\n""" * 4

# Tokenizar e verificar o tamanho
tokens_contexto = tokenizer(
    contexto_rag,
    return_tensors="pt",
    truncation=True,
    max_length=12000
)

n_tokens = tokens_contexto['input_ids'].shape[1]
print(f"{'='*50}")
print(f"PASSO 2 — Contexto RAG gerado:")
print(f"  → {len(contexto_rag):,} caracteres")
print(f"  → {n_tokens:,} tokens após tokenização")
print(f"{'='*50}")

## Passo 3: O Gargalo de Geração — SEM KV Cache (baseline)

In [ ]:
# --- BASELINE: Geração SEM KV Cache ---
# Desativar cache força o recálculo de Q, K, V a cada token gerado → O(n²) por passo

PROMPT_CLINICO = """Com base nos manuais médicos fornecidos, gere um resumo clínico estruturado
abordando os principais protocolos terapêuticos, achados laboratoriais relevantes e condutas
de emergência prioritárias:

CONTEXTO MÉDICO:
""" + contexto_rag[:3000]  # Usar porção do contexto para o prompt

N_NOVOS_TOKENS = 100

# Desativar KV Cache
model_base.config.use_cache = False

input_ids = tokenizer(
    PROMPT_CLINICO,
    return_tensors="pt",
    truncation=True,
    max_length=2048
).input_ids.to(model_base.device)

print(f"Gerando {N_NOVOS_TOKENS} tokens SEM KV Cache...")
print(f"Tokens no prompt: {input_ids.shape[1]}")
print("(Isso será LENTO — recálculo completo de Q,K,V a cada passo)\n")

torch.cuda.reset_peak_memory_stats()
torch.cuda.empty_cache()

t_inicio = time.time()

with torch.no_grad():
    output_sem_cache = model_base.generate(
        input_ids,
        max_new_tokens=N_NOVOS_TOKENS,
        do_sample=False,          # Greedy decoding
        use_cache=False,          # ← A PEGADINHA: sem cache
        pad_token_id=tokenizer.eos_token_id,
    )

t_sem_cache = time.time() - t_inicio
pico_vram_sem_cache = torch.cuda.max_memory_allocated() / 1024**2

texto_gerado = tokenizer.decode(
    output_sem_cache[0][input_ids.shape[1]:],
    skip_special_tokens=True
)

print(f"\n{'='*50}")
print("MÉTRICA 2 — Geração SEM KV Cache:")
print(f"  → Tempo total:     {t_sem_cache:.2f}s")
print(f"  → Tokens/segundo:  {N_NOVOS_TOKENS/t_sem_cache:.2f} tok/s")
print(f"  → Pico VRAM:       {pico_vram_sem_cache:.2f} MB")
print(f"\nTexto gerado (primeiros 200 chars):")
print(texto_gerado[:200])
print(f"{'='*50}")

## Passo 4: A Engenharia de Otimização — KV Cache + FlashAttention-2

In [ ]:
# --- OTIMIZADO: Recarregar modelo com FlashAttention-2 ---
# FlashAttention-2 usa SRAM (memória on-chip) para evitar acessos redundantes à HBM (VRAM)

print("Recarregando modelo com FlashAttention-2...")
del model_base
torch.cuda.empty_cache()

torch.cuda.reset_peak_memory_stats()

model_otimizado = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="flash_attention_2",  # ← FlashAttention-2 ativado
)
model_otimizado.eval()

# Ativar KV Cache
model_otimizado.config.use_cache = True

print("Modelo otimizado carregado com sucesso.")

# Reutilizar os mesmos input_ids
input_ids = tokenizer(
    PROMPT_CLINICO,
    return_tensors="pt",
    truncation=True,
    max_length=2048
).input_ids.to(model_otimizado.device)

print(f"\nGerando {N_NOVOS_TOKENS} tokens COM KV Cache + FlashAttention-2...")

torch.cuda.reset_peak_memory_stats()
torch.cuda.empty_cache()

t_inicio = time.time()

with torch.no_grad():
    output_otimizado = model_otimizado.generate(
        input_ids,
        max_new_tokens=N_NOVOS_TOKENS,
        do_sample=False,
        use_cache=True,           # ← KV Cache ativado
        pad_token_id=tokenizer.eos_token_id,
    )

t_com_otimizacao = time.time() - t_inicio
pico_vram_otimizado = torch.cuda.max_memory_allocated() / 1024**2

texto_otimizado = tokenizer.decode(
    output_otimizado[0][input_ids.shape[1]:],
    skip_special_tokens=True
)

print(f"\n{'='*50}")
print("MÉTRICA 3 — Geração COM KV Cache + FlashAttention-2:")
print(f"  → Tempo total:     {t_com_otimizacao:.2f}s")
print(f"  → Tokens/segundo:  {N_NOVOS_TOKENS/t_com_otimizacao:.2f} tok/s")
print(f"  → Pico VRAM:       {pico_vram_otimizado:.2f} MB")
print(f"\nTexto gerado (primeiros 200 chars):")
print(texto_otimizado[:200])
print(f"{'='*50}")

## Comparativo Final de Métricas

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# --- Dados comparativos ---
labels = ['Sem KV Cache\n(Baseline)', 'Com KV Cache\n+ FlashAttention-2']
tempos = [t_sem_cache, t_com_otimizacao]
vrames = [pico_vram_sem_cache, pico_vram_otimizado]
throughputs = [N_NOVOS_TOKENS/t_sem_cache, N_NOVOS_TOKENS/t_com_otimizacao]

cores = ['#e74c3c', '#2ecc71']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Laboratório 10 — Comparativo de Otimização de Inferência', fontsize=14, fontweight='bold')

# Gráfico 1: Tempo de Geração
bars0 = axes[0].bar(labels, tempos, color=cores, edgecolor='black', linewidth=0.8)
axes[0].set_title('Tempo de Geração (s)\n← Menor é melhor')
axes[0].set_ylabel('Segundos')
for bar, val in zip(bars0, tempos):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}s', ha='center', fontweight='bold')

# Gráfico 2: Pico de VRAM
bars1 = axes[1].bar(labels, vrames, color=cores, edgecolor='black', linewidth=0.8)
axes[1].set_title('Pico de VRAM (MB)\n← Menor é melhor')
axes[1].set_ylabel('Megabytes')
for bar, val in zip(bars1, vrames):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 f'{val:.0f} MB', ha='center', fontweight='bold')

# Gráfico 3: Throughput
bars2 = axes[2].bar(labels, throughputs, color=cores, edgecolor='black', linewidth=0.8)
axes[2].set_title('Throughput (tok/s)\n↑ Maior é melhor')
axes[2].set_ylabel('Tokens/segundo')
for bar, val in zip(bars2, throughputs):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{val:.1f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('benchmark_lab10.png', dpi=150, bbox_inches='tight')
plt.show()

# Resumo textual
speedup = t_sem_cache / t_com_otimizacao
reducao_vram = (1 - pico_vram_otimizado/pico_vram_sem_cache) * 100
print(f"\n{'='*50}")
print("RESUMO DO BENCHMARK:")
print(f"  Speedup de geração:      {speedup:.2f}x mais rápido")
print(f"  Redução de VRAM:         {reducao_vram:.1f}%")
print(f"  Ganho de throughput:     {throughputs[1]/throughputs[0]:.2f}x")
print(f"{'='*50}")

## Conclusão

Este laboratório demonstrou na prática como a combinação de **QLoRA (4-bits)**, **KV Cache** e **FlashAttention-2** resolve o problema de OOM em pipelines RAG com contextos massivos.

Consulte o `README.md` para a análise arquitetural completa (Passo 5).